In [1]:
import csv

# Read CSV file
with open("res/outagesNYISO.csv") as file:
    csv_reader = csv.DictReader(file)
    original_data = list(csv_reader)

with open("processed-actual-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    our_data = list(csv_reader)

In [2]:
import re
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo


PATTERN = re.compile(
    r"^DateObject\[\{\s*"
    r"(?P<year>-?\d+)\s*,\s*"
    r"(?P<month>\d+)\s*,\s*"
    r"(?P<day>\d+)\s*,\s*"
    r"(?P<hour>\d+)\s*,\s*"
    r"(?P<minute>\d+)\s*,\s*"
    r"(?P<second>\d+(?:\.\d*)?)\s*"
    r'\}\s*,\s*"[^"]*"\s*,\s*"[^"]*"\s*,\s*"(?P<tz>[^"]+)"\s*\]$'
)


def parse_original_dateobject(s: str) -> datetime:
    m = PATTERN.match(s.strip())
    if not m:
        raise ValueError(f"Invalid DateObject format: {s}")

    year = int(m.group("year"))
    month = int(m.group("month"))
    day = int(m.group("day"))
    hour = int(m.group("hour"))
    minute = int(m.group("minute"))
    sec_float = float(m.group("second"))  # handles 0. / 18. / 12.345
    tz = ZoneInfo(m.group("tz"))

    sec_int = int(sec_float)
    micro = int(round((sec_float - sec_int) * 1_000_000))
    if micro == 1_000_000:  # rounding edge case
        sec_int += 1
        micro = 0

    dt = datetime(year, month, day, hour, minute, 0, 0)
    dt += timedelta(seconds=sec_int, microseconds=micro)
    return dt

In [3]:
for datum in original_data:
    datum["PTID"] = int(datum["PTID"])
    datum["OutDatetime"] = parse_original_dateobject(datum["OutDatetime"])
    datum["MinTimeStamp"] = parse_original_dateobject(datum["MinTimeStamp"])
    datum["MaxTimeStamp"] = parse_original_dateobject(datum["MaxTimeStamp"])

original_data = sorted(original_data, key=lambda d: d["OutDatetime"])

In [4]:
original_data[:3]

[{'OutageID': '2',
  'PTID': 26007,
  'Name': 'GOWANUSA-GREENWD__138_42G24',
  'OutDatetime': datetime.datetime(2008, 11, 1, 0, 12),
  'MinTimeStamp': datetime.datetime(2008, 11, 1, 0, 17, 18),
  'MaxTimeStamp': datetime.datetime(2008, 11, 2, 23, 57, 19),
  'NearestSCtime': 'DateObject[{2008, 11, 1, 0, 12, 17.}, "Instant", "Gregorian", "America/New_York"]',
  'TimeDifferencefromScheduled': '0.2833333333333333',
  'OutageType': 'Planned',
  'DurationBAD': '2865.3166666666666',
  'LineNameOLD': '{"GOWANUSA", "GREENWD", 1}',
  'Voltage': '138',
  'BusNames': '{"GOWANUSA", "GREENWD"}',
  'UnitNumber': '1',
  'LineName': '{"GOWANUSA", "GREENWD", 1}',
  'OutAbstime': '3434501520',
  'TimeZone': 'EPT'},
 {'OutageID': '7',
  'PTID': 25048,
  'Name': 'JAMAICA_-VALLYSTR_138_901 L_M',
  'OutDatetime': datetime.datetime(2008, 11, 1, 1, 27),
  'MinTimeStamp': datetime.datetime(2008, 11, 1, 1, 32, 18),
  'MaxTimeStamp': datetime.datetime(2008, 11, 3, 23, 57, 18),
  'NearestSCtime': 'DateObject[{2008

In [5]:
for datum in our_data:
    datum["PTID"] = int(datum["PTID"])
    datum["OutDatetime"] = dt = datetime.strptime(
        datum["OutDatetime"],
        "%Y-%m-%d %H:%M:%S",
    )
    datum["MinTimeStamp"] = dt = datetime.strptime(
        datum["MinTimeStamp"],
        "%Y-%m-%d %H:%M:%S",
    )
    datum["MaxTimeStamp"] = dt = datetime.strptime(
        datum["MaxTimeStamp"],
        "%Y-%m-%d %H:%M:%S",
    )
    datum["FirstBus"] = datum["FirstBus"]
    datum["SecondBus"] = datum["SecondBus"]

our_data = sorted(our_data, key=lambda d: d["OutDatetime"])

In [6]:
our_data[:3]

[{'PTID': 26053,
  'Name': 'MOUNTAIN-SWANROAD_115_104-3',
  'OutDatetime': datetime.datetime(2005, 1, 1, 0, 0),
  'MinTimeStamp': datetime.datetime(2012, 9, 11, 11, 52, 17),
  'MaxTimeStamp': datetime.datetime(2013, 8, 8, 8, 37, 17),
  'Voltage': '115',
  'FirstBus': 'MOUNTAIN',
  'SecondBus': 'SWANROAD'},
 {'PTID': 25094,
  'Name': 'ANDOVER_-PALMITER_115_157_932',
  'OutDatetime': datetime.datetime(2005, 2, 1, 0, 0),
  'MinTimeStamp': datetime.datetime(2006, 3, 20, 9, 12, 22),
  'MaxTimeStamp': datetime.datetime(2015, 1, 14, 14, 22, 19),
  'Voltage': '115',
  'FirstBus': 'ANDOVER_',
  'SecondBus': 'PALMITER'},
 {'PTID': 25243,
  'Name': 'INGHAM_C-INGHAM_E_115_R81',
  'OutDatetime': datetime.datetime(2005, 2, 1, 0, 0),
  'MinTimeStamp': datetime.datetime(2006, 4, 19, 9, 2, 23),
  'MaxTimeStamp': datetime.datetime(2015, 1, 14, 14, 17, 18),
  'Voltage': '115',
  'FirstBus': 'INGHAM_C',
  'SecondBus': 'INGHAM_E'}]

In [7]:
# filter out our data
first_min_date = min(original_data, key=lambda x: x["MinTimeStamp"])["MinTimeStamp"]
last_min_date = max(original_data, key=lambda x: x["MinTimeStamp"])["MinTimeStamp"]
first_max_date = min(original_data, key=lambda x: x["MaxTimeStamp"])["MaxTimeStamp"]
last_max_date = max(original_data, key=lambda x: x["MaxTimeStamp"])["MaxTimeStamp"]
our_data_filtered = [
    row
    for row in our_data
    if first_min_date <= row["MinTimeStamp"] <= last_min_date
    and first_max_date <= row["MaxTimeStamp"] <= last_max_date
]

In [8]:
len(our_data_filtered), len(original_data)

(46144, 45178)

In [9]:
from pathlib import Path

import pandas as pd
from tqdm import tqdm


diff_list = []
counter = 0
for row in tqdm(our_data_filtered):
    is_in_original = False
    for counter, orow in enumerate(original_data):
        c1 = row["MinTimeStamp"] == orow["MinTimeStamp"]
        c2 = row["MaxTimeStamp"] == orow["MaxTimeStamp"]
        c3 = row["PTID"] == orow["PTID"]
        c4 = row["OutDatetime"] == orow["OutDatetime"]
        if c1 and c2 and c3 and c4:
            is_in_original = True
            break
    if not is_in_original:
        diff_list.append(row)
        counter += 1
print(counter)

csv_path = Path("diff-minmax-outages.csv")
pd.DataFrame(diff_list).to_csv(csv_path, index=False)
print(f"wrote {len(diff_list)} rows to {csv_path}")

100%|██████████| 46144/46144 [05:06<00:00, 150.73it/s]


45171
wrote 4478 rows to diff-minmax-outages.csv


# check graph connectivity

In [10]:
import igraph

our_bus_set = set([row[key] for row in our_data for key in ["FirstBus", "SecondBus"]])

print(f"num vertices: {len(our_bus_set)}")

bus_name_to_index = {
    bus_name: i for i, bus_name in enumerate(sorted(list(our_bus_set)))
}
index_to_bus_name = {
    i: bus_name for i, bus_name in enumerate(sorted(list(our_bus_set)))
}
n = len(bus_name_to_index)
g = igraph.Graph(n=len(bus_name_to_index), directed=False)

edge_set = set()
for datum in our_data:
    b1, b2 = datum["FirstBus"], datum["SecondBus"]
    b1_index, b2_index = bus_name_to_index[b1], bus_name_to_index[b2]
    edge_set.add((b1_index, b2_index))
g.add_edges(list(edge_set))

print(g.is_connected())

num vertices: 1456
False


In [11]:
our_bus_set_ = set(
    [
        row[key].strip("_").replace("_", "").replace(".", "")
        for row in our_data
        for key in ["FirstBus", "SecondBus"]
    ]
)

original_bus_set = set(
    [bus_name for row in original_data for bus_name in eval(row["BusNames"])]
)

In [12]:
original_bus_set - our_bus_set_

{'ARROWPH',
 'ASCTAP',
 'B31LSWJ',
 'B5DSWJ',
 'BAY1-4',
 'BEARD20',
 'BEARSWP',
 'BECKNM',
 'BRAYTPT',
 'BRUCEA',
 'BRUCEB',
 'CAMPVPF',
 'CARPHL',
 'CEDARGR',
 'CHIPPTP',
 'COLDRIV',
 'COMFDDC',
 'E131TAP',
 'EASTAVE',
 'EASYJ21',
 'EASYJ22',
 'FALLSVG',
 'GRGRTPJ',
 'HOMERC',
 'HOPECR',
 'IBMK21',
 'IBMK24',
 'LYNVILD',
 'NEBOQ24',
 'NEBOQ29',
 'NORHBR',
 'NORTHRD',
 'NRUTLND',
 'OSWALDJ',
 'PEATST',
 'PLUMTRE',
 'PRATTSJ',
 'QCITY',
 'RYTNJA',
 'RYTNJB',
 'SALBNJ',
 'SCOVRCK',
 'SHERO',
 'SOEND',
 'STA180',
 'TORRTRM',
 'TRMBJA',
 'TRMBJB',
 'VRNONVT',
 'VTYANK',
 'WESTBUS',
 'WMEDWAY'}

In [13]:
our_bus_set_ - original_bus_set

{'01BEDNGT',
 '01BLACK',
 '01PRNTY',
 '05COOK',
 '19WTRMN',
 '3 MILE I',
 'AFTON',
 'AIRPRDCT',
 'ALANBQ26',
 'ALANBQ28',
 'ALANBQ30',
 'ALBION31',
 'ALBIONRD',
 'ALLENFLS',
 'ALLISJE9',
 'AMSTERDM',
 'ANDES',
 'APWDJL24',
 'ARKVILLE',
 'ARROW PH',
 'ASC TAP',
 'ASHFELDS',
 'ASTORIAG',
 'AUSTINRD',
 'AXTELLRD',
 'B31L SWJ',
 'B5D SW J',
 'BABYLON',
 'BAGLEY',
 'BAY  1-4',
 'BEA RD20',
 'BEAR SWP',
 'BEARSWPT',
 'BECK NM',
 'BELLEAYR',
 'BELLVILE',
 'BERGEN 1',
 'BERLIN',
 'BLUESTON',
 'BNVLTAP',
 'BRANCOMB',
 'BRANTFRD',
 'BRAYT PT',
 'BREMEN',
 'BRENTWOD',
 'BRGHTWTR',
 'BRIDGPRT',
 'BRIDGWTR',
 'BRKSMFL',
 'BROCKVLE',
 'BROOKVLE',
 'BRUCE A',
 'BRUCE B',
 'BRUNSWCK',
 'BUNCE',
 'BURCHES',
 'BURCHESH',
 'BURDSTE',
 'BURDSTW',
 'BYRON',
 'CALEDNIA',
 'CALVRTON',
 'CAMPV PF',
 'CAMPVILL',
 'CANAL345',
 'CANTON',
 'CARP HL',
 'CARRCRCL',
 'CARVER',
 'CATAR X1',
 'CATARQUI',
 'CEDAR GR',
 'CEDAROR',
 'CEDARSHQ',
 'CEDRHRST',
 'CENTERST',
 'CHAT FOH',
 'CHESTER',
 'CHIPP TP',
 'CLARJV73',


In [19]:
comps = g.components(mode="weak")
print(len(comps))
print([len(comp) for comp in comps])
max_comp = max(comps, key=lambda x: len(x))
max_comp_bus_names = set([index_to_bus_name[comp] for comp in max_comp])

26
[1376, 2, 2, 2, 4, 8, 5, 2, 10, 2, 2, 4, 2, 5, 3, 2, 2, 2, 5, 3, 2, 2, 2, 3, 2, 2]


In [20]:
max_comp_bus_names

{'LEWISRN_',
 'BRK_SMFL',
 'LUDLUM_B',
 'GREENDGE',
 'GRUMMAN_',
 'PEROXYCM',
 'BOSTJN22',
 'BENSHRST',
 'VNWAGNER',
 'SWANRD_3',
 'WMPOTTER',
 'DALEROAD',
 'W.BABYLN',
 'GUARDIAN',
 'BREMEN__',
 'FITZWILL',
 'JOHNSTWN',
 'HARSNRAD',
 'N_FERRIS',
 'SUFOKAIR',
 'SABICO__',
 'WESTMLTN',
 'FORST-SE',
 'GLENDALE',
 'W MEDWAY',
 'BRUCE B_',
 'DEERFLD_',
 'ASTORIA5',
 '49STREET',
 'GOWANUSS',
 'WALDWICK',
 'COOPERNY',
 'CNTYLPRS',
 'ALBANY__',
 'LNDVFT__',
 'EAGLE___',
 'HOLTS_8G',
 'GETZVL60',
 'SUGRLOAF',
 'STA_82__',
 'BLUESTON',
 'CROSBY__',
 'LEVITTWN',
 'BETHLMRD',
 'SUGARISL',
 'MAINS_NY',
 'BEAR SWP',
 'FALLS VG',
 'STA_113_',
 'MORAINE_',
 'PINELAWN',
 'SHOEMAKR',
 'GOETHALS',
 'HMRSHM80',
 'BARRE___',
 'HAVERSTK',
 'LONGLANE',
 'MARK2J12',
 'NO.LEROY',
 'BARTELRD',
 'VINCO___',
 'PENHOTAP',
 'COLTON__',
 'MONSEY__',
 'LOCKPORT',
 'BECK NM_',
 'BERGEN 1',
 'CONASTON',
 'DENNISON',
 'MALONE__',
 'PLSTVYCH',
 'STONYBRK',
 'BOWLINOR',
 'SWAGRTWN',
 'ASC TAP_',
 'LENOX___',
 'WATKINRD',

# Filter the data by keeping only the largest connected component

In [ ]:
print(len(our_data))
our_data = [
    row
    for row in our_data
    if row["FirstBus"] in max_comp_bus_names and row["SecondBus"] in max_comp_bus_names
]
print(len(our_data))

74063
74063


In [25]:
import pandas as pd

actual_outage_csv_path = Path("processed-actual-outages.csv")

rows = sorted(our_data, key=lambda x: x["OutDatetime"])
df = pd.DataFrame(rows)

# write out; use the existing json path with a .csv suffix
df.to_csv(actual_outage_csv_path, index=False)

print(f"saved {len(df)} rows to {actual_outage_csv_path}")

saved 74063 rows to processed-actual-outages.csv


# Carrington/Dobson end dates are wrong!

In [16]:
for row in original_data:
    if row["MinTimeStamp"].month != row["MaxTimeStamp"].month:
        print(row)

In the Carrington/Dobson data, none of the outages span two months. They all end on the last day of the month.

example:

Ours:

100001824	CEDAR GR-HINCHMAN_230_E2257-2	2009-01-10 00:01:00	2009-01-14 11:27:18	2009-04-04 23:57:19	230	CEDAR GR	HINCHMAN

Carrington/Dobson:

100001824	CEDAR GR-HINCHMAN_230_E2257-2	{2009, 1, 10, 0, 1, 0.}	{2009, 1, 14, 11, 27, 18.}	{2009, 1, 31, 23, 57, 18.}